In [2]:
import os
from time import perf_counter

import httpx
import openai

API_KEY = os.getenv("MODEL_API_KEY_DEV")
if not API_KEY:
    raise RuntimeError(
        "Missing MODEL_API_KEY_DEV in environment. Restart the dev container after key generation/rotation."
    )

BASE_URL = os.getenv("MODEL_BASE_URL_CLEAN")
if not BASE_URL:
    raise RuntimeError(
        "Missing MODEL_BASE_URL_CLEAN in environment. Restart the dev container after env updates."
    )

CA_CERT_PATH = "/certs/.caroot/rootCA.pem"

verify_config = CA_CERT_PATH if os.path.exists(CA_CERT_PATH) else True
http_client = httpx.Client(verify=verify_config, timeout=30.0)

client = openai.OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    http_client=http_client,
)


def extract_response(response):
    if (
        hasattr(response, "choices")
        and response.choices
        and hasattr(response.choices[0], "message")
        and hasattr(response.choices[0].message, "content")
    ):
        return response.choices[0].message.content
    else:
        return ""


def query_models(model_list, query_function):
    for model in model_list:
        try:
            start = perf_counter()
            response = query_function(model)
            end = perf_counter()
            print(
                f"Model: {model} - Duration: {end - start:.2f}\n{extract_response(response)}\n\n"
            )
        except Exception as e:
            print(f"Error talking to {model}: {e}\n")

## Available models

### All models

In [3]:
models = client.models.list()
for model in models:
    print(model.id)

ollama_chat/deepseek-coder:6.7b
ollama_chat/deepseek-r1:7b
ollama_chat/llama3.1:8b
ollama_chat/llama3.2:1b
ollama_chat/llama3.2:3b
ollama_chat/mistral-nemo:12b
ollama_chat/mistral:7b
ollama_chat/qwen2.5-coder:7b
ollama_chat/qwen2.5:1.5b
ollama_chat/qwen3.5:0.8b
ollama_chat_stream/llama3.2:3b
ollama/nomic-embed-text:latest
ollama/qllama/bge-small-en-v1.5:q4_k_m
gemini-2.5-flash-lite
gemini-2.5-pro
groq-llama-3.1-8b-instant
groq-llama-3.3-70b
mistral-large
mistral-large-old
mistral-small


### Filter models: all chat models, reduced set for loop

In [3]:
chat_models = []
for model in models:
    if not any(substr in model.id for substr in ["code", "embed", "bge"]):
        chat_models.append(model.id)

print(f"Chat models: {chat_models}\n")

reduced_chat_models = []
for model in chat_models:
    if not any(
        substr in model
        for substr in [
            "ollama_chat/",
            "large",
            "pro",
        ]
    ):
        reduced_chat_models.append(model)


# I have a CPU only notebook env, and loading many models is slow
# if "ollama_chat/llama3.2:3b" in chat_models:
# reduced_chat_models.append("ollama_chat/llama3.2:3b")

# Ollama 3.5 0.8 is fast to produce token BUT due to its low knowledge it can given long answers
# if "ollama_chat/qwen3.5:0.8b" in chat_models:
# reduced_chat_models.append("ollama_chat/qwen3.5:0.8b")

print(f"Reduced chat models: {reduced_chat_models}\n")

## Q&A

In [4]:
def qa(model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are LLM 'superGPT 4.0' and developed by the company WORLD OF SUPERHEROES.",
            },
            {"role": "user", "content": "Please, introduce yourself."},
        ],
        timeout=600,
    )
    return response


query_models(chat_models, qa)

## Math

In [5]:
def mathexpert(model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a math expert."},
            {
                "role": "user",
                "content": "I give you eight apples. I take away three. How many apples do I have left?",
            },
            {"role": "user", "content": "Did you claim I have 5 apples left?"},
        ],
    )
    return response


query_models(reduced_chat_models, mathexpert)

## Spelling (via FatherPhi)

In [6]:
def spellcheck(model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a spelling expert."},
            {
                "role": "user",
                "content": "Which number between 1 and 100 has an 'a' in it?",
            },
        ],
    )
    return response


query_models(reduced_chat_models, spellcheck)

## Physical world

In [7]:
def physicalworld(model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a physical world expert."},
            {
                "role": "user",
                "content": "What is left from a triangle after removing a corner?",
            },
        ],
    )
    return response


query_models(reduced_chat_models, physicalworld)

## Cooking myself

In [8]:
def chef(model):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a cooking expert."},
            {
                "role": "user",
                "content": "Give me a recipe for cooking a soup made of myself! I am a human. How long do I need to boil myself?",
            },
        ],
    )
    return response


query_models(reduced_chat_models, chef)